<a href="https://colab.research.google.com/github/NBekhruzbek/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers bitsandbytes datasets peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 28.7 MB/s eta 0:00:00


In [1]:
!pip install -U "torchao>0.16.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
import torchao
from datasets import load_dataset
from trl import SFTTrainer

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens map', tokenizer.special_tokens_map)


# ---------------------------------------------------------------
# QUANTIZATION (kvantlash)
# Model og'irliklarini (weights) 16-bit o'rniga 4-bitda saqlaymiz.
# Natija: GPU xotirasi ~3-4 barobar kamayadi (~2.2 GB -> ~0.7 GB),
# aniqlik esa juda oz pasayadi.
# QLoRA fine-tuning uchun asos bo'ladi.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # og'irliklarni 4-bitda yuklash
    bnb_4bit_use_double_quant=True,         # scale konstantalarini ham kvantlash (qo'shimcha xotira tejash)
    bnb_4bit_compute_dtype=torch.bfloat16,  # hisoblashlar bfloat16 da bajariladi (saqlash 4-bit, hisob 16-bit)
    bnb_4bit_quant_type='nf4'               # NF4 (NormalFloat4): normal taqsimotdagi og'irliklar uchun optimal format
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    dtype=torch.bfloat16
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is."
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)

first_block = model.model.layers[0]
print('frist_block:', first_block)
print('================================')
print(first_block.self_attn)
print('================================')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is.
frist_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
  (v_proj): Linear4bit(in_features=2048, out_feat

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


## datasets library | load_dataset
* insturction tuning

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split='train')
dataset
dataset[0]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Input:\n'
        f"{input_text}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        '### Response:\n'
        f"{output_text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return{"text": generate_prompt(example)}

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset[0]["text"]

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n\n'

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42) # seed =>  har safar bir xil malumotlarni shuffle qilib oladi

In [ ]:
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1]
# [7, 3, 2, 8, 5, 6, 9, 4, 0, 1] EXACT SAME ORDER

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning => PEFT
# PEFT => Parameter-Efficent Fine-Tuning => Kerekli qism fine tuning qilinadi
# OOM => Out of Memory

In [ ]:
from peft import LoraModel, LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8, # rank -> lora adapter hajmi
    lora_alpha=16, # scaling
    lora_dropout=0.05, # ourfitni kamaytirish
    task_type="CAUSAL_LM", # TinyLamaga mos type
    target_modules=['q_proj', 'v_proj'],
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLoRa
# LoRa

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # VRAM
    num_train_epochs=2, #overfit
    logging_steps=20,
    output_dir='./jaydari_gpt',
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    report_to="none"
)

In [ ]:
print(dataset.column_names)

['output', 'input', 'instruction', 'text']


In [43]:
# SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x["text"],
    args=training_args
)

trainer.train()
model.save_pretrained("jaydari_gpt")
tokenizer.save_pretrained("jaydari_gpt")

Step,Training Loss
20,1.545659
40,1.593953
60,1.328738
80,1.372932
100,1.293833
120,1.309018
140,1.333240
160,1.233405
180,1.294806
200,1.207896


Step,Training Loss
20,1.545659
40,1.593953
60,1.328738
80,1.372932
100,1.293833
120,1.309018
140,1.333240
160,1.233405
180,1.294806
200,1.207896


('jaydari_gpt/tokenizer_config.json',
 'jaydari_gpt/chat_template.jinja',
 'jaydari_gpt/tokenizer.json')

In [50]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto"
)

model_base.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device) # GPU, CPU

with torch.no_grad():
  output_base = model_base.generate(
      **inputs_base,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True
  )

print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))


#FINE-TUNED MODEL INFERENCE

In [5]:
model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device) # GPU, CPU

with torch.no_grad():
  output_ft = model_ft.generate(
      **inputs_ft,
      max_new_tokens=120,
      temperature=0.7,
      do_sample=True
  )

print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_base[0], skip_special_tokens=True))
